In [ ]:
import os
import shutil
from pathlib import Path

# Auto-discover the path
DATASET_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "pretrain.py" in files:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError("Could not find 'pretrain.py' in any /kaggle/input/ structure. Is the dataset attached securely?")

print(f"✓ Auto-detected SBERTa codebase at: {DATASET_PATH}")

# Copy the core python packages and scripts
shutil.copytree(f"{DATASET_PATH}/sberta", "/kaggle/working/sberta", dirs_exist_ok=True)
shutil.copy(f"{DATASET_PATH}/pretrain.py", "/kaggle/working/pretrain.py")

# Bring over root tokenizer files if they have been generated
for f in ["sberta.model", "sberta.vocab", "train_tokenizer.py"]:
    if Path(f"{DATASET_PATH}/{f}").exists():
        shutil.copy(f"{DATASET_PATH}/{f}", f"/kaggle/working/{f}")

# CRITICAL: Copy corpus to local SSD for 2-3x faster I/O
LOCAL_CORPUS = "/kaggle/working/corpus"
if not Path(LOCAL_CORPUS).exists():
    print("🚀 Copying corpus to local SSD (speeds up DataLoader streaming)...")
    shutil.copytree(f"{DATASET_PATH}/corpus", LOCAL_CORPUS)
    corpus_size = sum(f.stat().st_size for f in Path(LOCAL_CORPUS).rglob("*.txt")) / 1e6
    print(f"✓ Copied {corpus_size:.1f} MB to local SSD")
else:
    print("✓ Corpus already on local SSD")

print(f"\n📊 Environment:")
print(f"  Codebase: {DATASET_PATH}")
print(f"  Corpus:   {LOCAL_CORPUS}")
print(f"  Output:   /kaggle/working/runs")


In [ ]:
# RTX 6000 Pro (94.5 GB VRAM) — optimized for maximum throughput
# Base config: 124M params → ~500 MB model + ~1.5 GB optimizer state
# At batch=512, grad_accum=4: effective_batch=2048, ~60 GB VRAM, ~30 GB headroom

cmd = f"""python /kaggle/working/pretrain.py \
    --config base \
    --corpus-dirs /kaggle/working/corpus \
    --tokenizer-dir {DATASET_PATH}/runs/tokenizer \
    --runs-dir /kaggle/working/runs \
    --run-id sberta-base-100k \
    --total-steps 100000 \
    --warmup-steps 10000 \
    --batch-size 512 \
    --grad-accum 4 \
    --lr 1e-4 \
    --max-length 512 \
    --checkpoint-every 5000 \
    --log-every 100"""

print(f"🚀 RTX 6000 Pro training :\n{cmd}\n")
!{cmd}
